### ランダムフォレストの説明変数重要性

In [ ]:
import pandas as pd
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)


In [ ]:
import warnings
import os
# warning messageを出さない
warnings.filterwarnings('ignore')

os.makedirs("image_executed", exist_ok=True)

In [ ]:
def get_data():
    """load data from a file.

    Returns:
        pd.DataFrame: data.
        [str]: explanatory variable names.
        str: target variable name
    """
    filename = "../data/TC_ReCo_detail_descriptor.csv"
    df = pd.read_csv(filename)
    descriptor_names = ['C_R', 'C_T', 'vol_per_atom', 'Z', 'f4', 'd5', 'L4f', 'S4f', 'J4f',
                        '(g-1)J4f', '(2-g)J4f']
    target_name = 'Tc'
    return df, descriptor_names, target_name


g_df, g_descriptor_names, g_target_name = get_data()




random forest回帰ではプリプロセスを行わなくても回帰結果は変わらないが行っておく。


In [ ]:
from sklearn.preprocessing import StandardScaler


def make_Xy(df, descriptor_names, targetlabel):
    """make X and y

    Args:
        df (pd.DataFrame): data
        descriptor_names (list): a list of features
        targetlabel (str): target name

    Returns:
        np.array: descriptor
        np.array: target values
    """
    scaler = StandardScaler()
    Xraw = df[descriptor_names].values
    scaler.fit(Xraw)
    X = scaler.transform(Xraw)
    y = df[targetlabel].values
    return X, y


g_X, g_y = make_Xy(g_df, g_descriptor_names, g_target_name)


規格化データの可視化をしておく。

random forest回帰を行う。

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score


reg.feature_importances_にはimpurity importanceと呼ばれる値が入っている。

まずこの値を表示する。

In [ ]:
g_rf = RandomForestRegressor(n_estimators=100, random_state=1)
g_rf.fit(g_X, g_y)


In [ ]:
g_rf.feature_importances_


In [ ]:
def show_importance(rf, descriptor_names,):
    """visualize importance

    Args:
        rf (RandomForestRegressor): Random Forest regression model.
        descriptor_names ([str]): a list of explanatory variable names.
    """
    fig, ax = plt.subplots()
    df_rf_imp = pd.DataFrame(
        {"label": descriptor_names, "importance": rf.feature_importances_})
    df_rf_imp.sort_values(by="importance", ascending=False, inplace=True)
    df_rf_imp.plot.bar(x="label", y="importance", ax=ax)
    fig.tight_layout()
    fig.savefig("image_executed/RECo_RF_impurity_importance.png")
    fig.show()


show_importance(g_rf, g_descriptor_names, )


### permutation importance


In [ ]:
from sklearn.inspection import permutation_importance

g_feature_importance = permutation_importance(
    g_rf, g_X, g_y, n_repeats=30, random_state=20)


feature_importanceには以下の結果が入る。

In [ ]:
g_feature_importance.keys()


importancesは各n_repeats回のscore減少が入る。下では３０回分の結果が入る。

In [ ]:
g_df_perm = pd.DataFrame(
    g_feature_importance["importances"], index=g_descriptor_names).T
g_df_perm


np.argsort(score_mean)[::-1]で平均値が大きい順に並べ替える。

In [ ]:
def show_boxplot(df):
    """show values as boxplot.

    Args:
        df (pd.DataFrame): data.
    """
    score_mean = np.mean(df.values, axis=0)
    iorder = np.argsort(score_mean)[::-1]
    fig, ax = plt.subplots()
    df.iloc[:, iorder].boxplot(rot=90, ax=ax)
    ax.set_ylabel("$R^2$ decrease")
    fig.tight_layout()
    fig.savefig("image_executed/RECo_RF_permutation_importance.png")
    fig.show()


show_boxplot(g_df_perm)


permutation importanceは回帰手法に依らないので線形回帰についてpermutation importanceを行う。

### 線型回帰係数
まず、比較のための線型回帰係数を表示しておく。

In [ ]:
from sklearn.linear_model import LinearRegression
g_lr = LinearRegression(fit_intercept=True)
g_lr.fit(g_X, g_y)


In [ ]:
def show_lr_coef(lr, descriptor_names):
    """show coefficients of linear regression model.

    Args:
        lr (LinearRegression): linear model.
        descriptor_names ([str]): a list of explnatory variable names.
    """
    df_lr_coef = pd.DataFrame(
        {"label": descriptor_names, "abs(coef)": np.abs(lr.coef_)})
    df_lr_coef.sort_values(by="abs(coef)", ascending=False, inplace=True)
    fig, ax = plt.subplots()
    df_lr_coef.plot.bar(x="label", y="abs(coef)", ax=ax)
    fig.savefig("image_executed/coef.png")
    fig.show()


show_lr_coef(g_lr, g_descriptor_names)


permutation importanceを計算する。

注意 :

$R^2$の最大値は１ですが、最小値は$-\infty$を取りえます。このため$R^2$の差は１よりも大きくなりえます。


In [ ]:
g_feature_importance = permutation_importance(
    g_lr, g_X, g_y, n_repeats=30, random_state=20)
g_df_lr_per = pd.DataFrame(
    g_feature_importance["importances"], index=g_descriptor_names).T


def show_r2_decrease(df_lr_per):
    """show leave-one-out R2 values.

    Args:
        df_lr_per (pd.DataFrame): data
    """
    score_mean = np.mean(df_lr_per.values, axis=0)
    iorder = np.argsort(score_mean)[::-1]
    fig, ax = plt.subplots()
    df_lr_per.iloc[:, iorder].boxplot(rot=90, ax=ax)
    ax.set_ylabel("$R^2$ decrease")


show_r2_decrease(g_df_lr_per)


In [ ]:
from sklearn.linear_model import RidgeCV
g_rdg = RidgeCV(fit_intercept=True)
g_rdg.fit(g_X, g_y)


In [ ]:
def show_abs__coef(rdg, descriptor_names):
    """show absolute values of coefficients of linear regression.

    Args:
        rdg (LinearRegression): linear regression model.
        descriptor_names ([str]): a list of explanatory variable names.
    """
    df_rdg_coef = pd.DataFrame(
        {"label": descriptor_names, "abs(coef)": np.abs(rdg.coef_)})
    df_rdg_coef.sort_values(by="abs(coef)", ascending=False, inplace=True)
    df_rdg_coef.plot.bar(x="label", y="abs(coef)",)


show_abs__coef(g_rdg, g_descriptor_names)


In [ ]:
def show_feature_importance(rdg, X, y, descriptor_names):
    """show feature importance.

    Args:
        rdg (RandomForestRegressor): Random Forest model.
        X (np.ndarray): explanatory variables.
        y (np.ndarray): target variables.
        descriptor_names ([str]): a list of explanatory variable names.
    """
    feature_importance = permutation_importance(
        rdg, X, y, n_repeats=30, random_state=20)
    df_rdg_per = pd.DataFrame(
        feature_importance["importances"], index=descriptor_names).T
    score_mean = np.mean(df_rdg_per.values, axis=0)
    iorder = np.argsort(score_mean)[::-1]

    df_rdg_per.iloc[:, iorder].boxplot(rot=90)
    plt.ylabel("$R^2$ decrease")


show_feature_importance(g_rdg, g_X, g_y, g_descriptor_names)


コメント

permutation importance, impurity importanceともに「ある一つのモデル」でのfeature importanceの評価です。


#### 問題

データを変える。
